In [1]:
from datetime import datetime
import re
from treetime import TreeTime, TreeTimeError, utils
from dendropy import Tree
from Bio import SeqIO
import baltic as bt
import os
import random
import json

In [26]:
clades = ['na_avian', 'eurasian_avian', 'swine', 'canine', 'human', 'equine']

In [3]:
host_trees = {}

for clade in clades:
    
    path = f'hosts/{clade}/summary_baltic.nwk'
    mytree= bt.loadNewick(path, absoluteTime= False)

    
    host_trees[clade] = mytree

In [ ]:
def parse_dates(aln_path: str):
    records = SeqIO.parse(aln_path, 'fasta')
    dates = {}
    for record in records:
        name = record.name
        # date_str = name.split('|')[-1]
        date = None
        for token in name.split('|'):
            if re.fullmatch(r'[\d\-/]{4,}', token) and not re.fullmatch(r'\d{5,}', token):
                if token.count('/') == 2:
                    date = datetime.strptime(token, '%m/%d/%Y')
                elif token.count('/') == 1:
                    date = datetime.strptime(token, '%m/%Y')
                elif token.count('-') == 2:
                    date = datetime.strptime(token, '%Y-%m-%d')
                elif token.count('-') == 1:
                    date = datetime.strptime(token, '%Y-%m')
                else:
                    date = datetime.strptime(token, '%Y')
        if not date:
            pass
        else:
            dec_date = date.year + ((date.month - 1) * 30 + date.day) / 365.0
            dates[name] = dec_date
    return dates

def estimate_clock_rate(tree_path: str, aln_path: str, outdir='.') -> (float, float):
    
    # This code was adapted from the  'estimate_clock_model' method in the treetime/wrappers.py.
    tree: Tree = Tree.get(path=tree_path, schema='newick', preserve_underscores=True)
    if len(tree.leaf_nodes()) > 1000:
        tree_path = os.path.join(outdir, os.path.split(tree_path)[-1] + '.sample1k.tre')
        taxa_labels = [t.label for t in tree.taxon_namespace]
        random.shuffle(taxa_labels)
        subtree: Tree = tree.extract_tree_with_taxa_labels(taxa_labels[:1000])
        subtree.write(path=tree_path, schema='newick')

    dates = parse_dates(aln_path)
    
    timetree = TreeTime(dates=dates, tree=tree_path, aln=aln_path, gtr='JC69', verbose=-1)  # TODO: JC->GTR?
          
    timetree.clock_filter(n_iqd=3, reroot='least-squares')
    timetree.use_covariation = True
    timetree.reroot()
    timetree.get_clock_model(covariation=True)
    
    timetree.run(root='least-squares', resolve_polytomies=True)

    inferred_root_date = timetree.tree.root.numdate
    print(f"inferred root date: {inferred_root_date:.4f}")

    r_val = timetree.clock_model['r_val']
    
    d2d = utils.DateConversion.from_regression(timetree.clock_model)
    clock_rate = round(d2d.clock_rate, 7)
    
    return d2d.clock_rate, r_val, inferred_root_date
    

In [31]:
host_stats = {}

for clade in clades:
    
    print(clade)
    
    clock_rate, r_val, root_yr = estimate_clock_rate(f"hosts/{clade}/summary_baltic.nwk", f"hosts/{clade}/h3nx_ha.fasta")
    
    host_stats[clade] = {
    'clock_rate': clock_rate,
    'root_yr': root_yr
    }

na_avian
inferred root date: 1932.7973
eurasian_avian
inferred root date: 1925.5975
swine
inferred root date: 1965.3745
canine
inferred root date: 2004.8255
human
inferred root date: 1967.4179
equine
inferred root date: 1954.4550


In [32]:
print(host_stats)

{'na_avian': {'clock_rate': 0.0014153628923017929, 'root_yr': 1932.7972743811047}, 'eurasian_avian': {'clock_rate': 0.0010022429427303807, 'root_yr': 1925.5975498054086}, 'swine': {'clock_rate': 0.003923517661115446, 'root_yr': 1965.3744788462288}, 'canine': {'clock_rate': 0.0021760366926091234, 'root_yr': 2004.8254515694057}, 'human': {'clock_rate': 0.00417372439954042, 'root_yr': 1967.4178682706547}, 'equine': {'clock_rate': 0.001865099192944831, 'root_yr': 1954.4550337364963}}


In [33]:
with open("host_clock_rates.json", "w") as f:
    json.dump(host_stats, f, indent=2)

In [ ]:
# checking tree lengths to assess why eurasian avian lineage has more reassortments but lower rate
# its because of the molecular clock

clades = ['eurasian_avian', 'swine']

rea_events = {"swine" : 442, "eurasian_avian" : 647}

for clade in clades:
    
    tree_length = sum(node.length for node in host_trees[clade].Objects)
    
    rate_div = rea_events[clade] / tree_length # reassortments per substitutions per site
    
    print(clade, tree_length, rate_div)
    

eurasian_avian 6.408299999999992 100.9628138507874
swine 11.785760000000046 37.50288483729503
